In [9]:
suppressMessages({
    library(dplyr)
    library(parallel)
library(ggplot2)
library(tidyr)
library(hydroGOF)
    })

In [10]:
gauge_df=readRDS('/nas/cee-water/cjgleason/colin/analyze confluence runs/SVS_df.rds')
gauged_reaches=unique(gauge_df$reach_id)
swot_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/swot/'
sos_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/sos/'
reach_ids=gauged_reaches[gauged_reaches %in% substr(list.files(swot_base),1,11)]

In [11]:
#get gauges in oceania
oceania_index=which(substr(reach_ids,1,1)=='5')
length(oceania_index)

[1] 206

In [31]:
#save a test case for AWS testing
#needs:
    #SWORD
    #Priors
    #SWOT input data
this_reach_id='56424600151'
sword='/nas/cee-ice/data/SWORD/SWORDv16/netcdf/oc_sword_v16.nc'
priors=paste0(sos_base,'oc_sword_v16_SOS_priors.nc')
swot=paste0(swot_base,this_reach_id,'_SWOT.nc')

In [32]:
file.path(sword)

[1] "/nas/cee-ice/data/SWORD/SWORDv16/netcdf/oc_sword_v16.nc"

In [35]:
file.copy(from = swot, to = file.path('/nas/cee-water/cjgleason/colin/BUSBOI/unit test/'))

[1] TRUE

In [15]:
run= substr(list.files(output_path),1,11)
unrun = !(reach_ids %in% run)
sum(unrun)

[1] 6

In [17]:
i

[1] 12

In [19]:
reach_ids[oceania_index][12]

[1] "56424600151"

In [40]:
source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')
# suppressWarnings({
for (i in 12){
output_path='/nas/cee-water/cjgleason/colin/BUSBOI/debug tests/TEST_FIXED_BED_RETRY/'
test= main_function(reach_ids[oceania_index][i],
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                   Q_prior='monthly', #'daily' or 'monthly'
                   tulip='OFF', #'ON' or 'OFF'
                   GVF_on=0, # 0 or 1
                   fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
}

# })

# a= Sys.time()
# clust=makeCluster(4)
# test=parLapply(clust,reach_ids[unrun],main_function,
#                    output_path=output_path,
#                    swot_base=swot_base,
#                    sos_base=sos_base,
#                    Q_prior='monthly', #or 'monthly'
#                    tulip='OFF', #or 'OFF'
#                    GVF_on=0,
#                    fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
# stopCluster(clust)
# print(Sys.time()-a)

In [63]:
###to control where the slurm files are written
working_dir='/nas/cee-water/cjgleason/colin/BUSBOI/debug_logs/'
setwd(working_dir)

source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')

library(rslurm, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)
library(whisker, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)

output_path='/nas/cee-water/cjgleason/colin/BUSBOI/tests/TEST_FIXED_BED_RETRY/'

testname='FBretry'
#slurm block
slurm_options= list(mem=128000, 'time'='50:00:00', #options for memory, time, parition, and an error file
                    partition ='ceewater_cjgleason-cpu',
                    error='slurm-%A_%a.err')
                    # exclude='ceewater-cpu009')
sjob <- slurm_map(as.list(reach_ids), #thing you want to loop over. must be a list
                  main_function,  # name of the function. declared above with the 'source' command
                  jobname = testname, # defined above. just for convenience
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                  Q_prior='monthly',
                  tulip='OFF',
                  GVF_on=0,
                  fix_bed=0,  #0 = 5pts, 1= 1pt, 2 = fixed
                  nodes = 1, # many nodes do you want?
                  preschedule_cores=FALSE, # keep this FALSE
                  cpus_per_node = 100, # how many CPUS per node. 
                  submit = TRUE, # if TRUE, submits to the cluster
                  slurm_options=slurm_options, #defined above
                  libPaths="/nas/cee-water/cjgleason/r-lib/" ) #library paths

Submitted batch job 51055114



In [35]:
length(list.files('/nas/cee-water/cjgleason/colin/BUSBOI/tests/monthly_nogvf_notulip_2s_changezo'))

[1] 1520